In [1]:
import optuna
import torch
from pytorch_lightning import Trainer
from pytorch_lightning.callbacks import EarlyStopping, ModelCheckpoint
from pytorch_lightning.loggers import TensorBoardLogger
from torchmetrics import F1Score
import warnings
warnings.filterwarnings('ignore')

from GradientGang.Pipeline.DataLoader.DataLoader import DataModule
from GradientGang.Pipeline.Architectures.Direct import Direct

In [4]:
# Fixed data loading parameters
data_params = {
    'data_dir': "../dataset/PirateProcessed/",
    'train_file_name': "pirate_pain_train.csv",
    'train_file_name_labels': "pirate_pain_train_labels.csv",
    'test_file_name': "pirate_pain_test.csv",
    'batch_size': 32,
    'num_workers': 0,
    'val_split': 0.2,
    'shuffle': True,
}

# Initialize data module
dataLoader = DataModule(params=data_params)
dataLoader.setup(stage='fit', includeTestInTrain=True)

trainLoader = dataLoader.train_dataloader()
valLoader = dataLoader.val_dataloader()

print("Data loaders initialized successfully!")
print(f"Training batches: {len(trainLoader)}")
print(f"Validation batches: {len(valLoader)}")

Data loaders initialized successfully!
Training batches: 58
Validation batches: 5


In [ ]:
def setUpEncoder(trial:optuna.Trial, architectureParameters:dict, datasetInfo:dict):
    # Setup global features encoder
    globalEmbeddingDim = 1
    globalEncoderParams = {
        "activation_function": "LeakyReLU",
        "layer_type": [
            {
                "name": "Linear",
                "params": {
                    "in_features": datasetInfo["numGlobalFeatures"],
                    "out_features": globalEmbeddingDim,
                    "bias": True,
                }
            },
        ]
    }
    architectureParameters["globalEncoderParams"] = globalEncoderParams
    architectureParameters["globalEmbeddingDim"] = globalEmbeddingDim


    # Setup time series encoder
    architectureType = trial.suggest_categorical("architectureType", ["Recurrent"])
    architectureParameters["architectureType"] = architectureType
    
    timeSeriesEncoderParams = {}

    if architectureType == "Recurrent":
        rnnType = trial.suggest_categorical("rnnType", ["LSTM", "GRU"])
        hiddenDim = trial.suggest_int("hiddenDim", 16, 128)
        numLayers = trial.suggest_int("numLayers", 1, 3)
        bidirectional = trial.suggest_categorical("bidirectional", [False, True])
        dropout = trial.suggest_float("recurrentDropout", 0.0, 0.5)
        activationFunction = trial.suggest_categorical("fcActivationFunction", ["ReLU", "LeakyReLU", "GELU"])
        timeSeriesEncoderParams = {

        }
        architectureParameters["EncoderParams"] = timeSeriesEncoderParams


        

    

In [ ]:
def objective(trial: optuna.trial.Trial) -> float:
    """
    Objective function for Optuna optimization.
    Returns validation F1 score to maximize.
    """
    macroArchitecture = trial.suggest_categorical("MacroArchitecture", ["Direct", "Autoencoder"])

    # Setup data
    includeTestInTrain = macroArchitecture == "Autoencoder"
    dataLoader.setup(stage='fit', includeTestInTrain=includeTestInTrain)
    trainLoader = dataLoader.train_dataloader()
    valLoader = dataLoader.val_dataloader()
    datasetInfo = dataLoader.getDatasetInfo()

    # Suggest common parameters
    archParams = {
        "MacroArchitecture": macroArchitecture,
        "LearningRate": trial.suggest_float("LearningRate", 1e-5, 1e-2, log=True),
        "RegularizationWeights": trial.suggest_float("RegularizationWeights", 1e-5, 1e-2, log=True),
        "ClassWeightsPath": "../dataset/PirateProcessed/class_weights.yaml",
        "OutputDim": 3,
    }





    
    # Generate architecture parameters from trial
    architecture_params = create_architecture_params(trial)
    
    # Create model
    model = Direct(architecture_params)
    
    # Training parameters
    max_epochs = 300
    
    # Add early stopping callback
    early_stopping_callback = EarlyStopping(
        monitor='val_F1',
        patience=architecture_params["Patience"],
        mode='max',  # We want to maximize F1 score
        verbose=False
    )

    # Save best model checkpoint
    checkpoint_callback = ModelCheckpoint(
        monitor='val_F1',
        mode='max',
        save_top_k=5,
        filename='autoencoder-best-{epoch:02d}-{val_F1:.3f}',
        verbose=True
    )

    trainer = Trainer(
        max_epochs=max_epochs,
        enable_progress_bar=False,
        enable_model_summary=False,
        log_every_n_steps=20,
        callbacks=[early_stopping_callback, checkpoint_callback],
    )
    
    # Train the model
    try:
        trainer.fit(model, trainLoader, valLoader)
        trainer.validate(model, valLoader)
    except Exception as e:
        print(f"Trial {trial.number} failed with error: {e}")
        return 0.0
    
    # Evaluate on validation set
    model.eval()
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for batch in valLoader:
            x, labels = batch
            preds = model(x)
            all_preds.extend(preds.argmax(dim=1).cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    # Calculate F1 score
    f1_metric = F1Score(task="multiclass", num_classes=3, average='weighted')
    f1_score = f1_metric(torch.tensor(all_preds), torch.tensor(all_labels))
    
    # Report intermediate value for pruning
    trial.report(f1_score.item(), step=trainer.current_epoch)
    
    # Handle pruning
    if trial.should_prune():
        raise optuna.TrialPruned()
    
    return f1_score.item()